In [ ]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.encoder import DisentangledEncoder
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    train_iql_from_loader,
    save_metrics_json,
)

In [ ]:
def rbf_kernel(x, sigma=1.0):
    dist = torch.cdist(x, x, p=2.0) ** 2
    return torch.exp(-dist / (2.0 * (sigma ** 2)))


def _center_kernel(K):
    row_mean = K.mean(dim=1, keepdim=True)
    col_mean = K.mean(dim=0, keepdim=True)
    total_mean = K.mean()
    return K - row_mean - col_mean + total_mean


def hsic_loss(z1, z2, sigma=1.0):
    """
    Compute the Hilbert-Schmidt Independence Criterion (HSIC) between two representations.

    Minimizing HSIC encourages z_task and z_irrel to become statistically independent.
    Uses double-centering via element-wise operations (avoids explicit n×n H matrix).
    """
    n = z1.size(0)
    K = rbf_kernel(z1, sigma=sigma)
    L = rbf_kernel(z2, sigma=sigma)
    Kc = _center_kernel(K)
    Lc = _center_kernel(L)
    return (Kc * Lc).sum() / ((n - 1) ** 2)

In [ ]:
# Experiment configuration
# INDEP_WEIGHT is read from src/experiment_config (env var INDEP_WEIGHT, default 0.05)
METHOD = "disentangled_hsic_indep_sweep"

def noise_tag(noise_dim, noise_scale, noise_type):
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

def iw_tag(indep_weight):
    return "iw_" + str(indep_weight).replace(".", "p")

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
IW_TAG = iw_tag(INDEP_WEIGHT)
SEED_TAG = f"seed_{SEED}"

CKPT_DIR    = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG / IW_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG / IW_TAG
# obs_stats depends only on (env, noise, seed), not on indep_weight — shared across all iw runs
OBS_DIR     = OBS_STATS_DIR   / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:",       DEVICE)
print("ENV_NAME:",     ENV_NAME)
print("NOISE_TAG:",    NOISE_TAG)
print("INDEP_WEIGHT:", INDEP_WEIGHT)
print("IW_TAG:",       IW_TAG)
print("CKPT_DIR:",     CKPT_DIR)
print("METRICS_DIR:",  METRICS_DIR)
print("OBS_DIR:",      OBS_DIR)

In [ ]:
# Dataset and dataloader
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

state_dim      = dataset.noisy_obs.shape[1]
action_dim     = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim
LATENT_DIM     = int(max(true_state_dim, NOISE_DIM) * 1.5)

np.savez(
    OBS_DIR / "obs_stats.npz",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)

print("state_dim (noisy):", state_dim)
print("true_state_dim:",    true_state_dim)
print("action_dim:",        action_dim)
print("LATENT_DIM:",        LATENT_DIM)

In [ ]:
# Pretrain the disentangled encoder, tracking per-epoch loss components
torch.manual_seed(SEED)
np.random.seed(SEED)

encoder = DisentangledEncoder(
    state_dim=state_dim,
    action_dim=action_dim,
    true_state_dim=true_state_dim,
    latent_dim=LATENT_DIM,
).to(DEVICE)

optim = torch.optim.Adam(encoder.parameters(), lr=3e-4)

pretrain_loader = DataLoader(dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
                             num_workers=4, pin_memory=True, persistent_workers=True)

pretrain_history = []

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    buf_total, buf_dyn, buf_rew, buf_indep = [], [], [], []

    for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in pretrain_loader:
        obs           = obs.to(DEVICE)
        act           = act.to(DEVICE)
        next_obs      = next_obs.to(DEVICE)
        rew           = rew.to(DEVICE)
        done          = done.to(DEVICE)
        pure_obs      = pure_obs.to(DEVICE)
        pure_next_obs = pure_next_obs.to(DEVICE)

        z_task, z_irrel = encoder(obs)

        pred_next_true = encoder.state_predictor(torch.cat([z_task, act], dim=-1))
        loss_dyn = torch.nn.functional.mse_loss(pred_next_true, pure_next_obs)

        pred_rew = encoder.reward_predictor(z_task)
        loss_rew = torch.nn.functional.mse_loss(pred_rew.squeeze(-1), rew)

        loss_hsic = hsic_loss(z_task, z_irrel, sigma=1.0)
        loss_indep_weighted = INDEP_WEIGHT * loss_hsic

        loss = loss_dyn + loss_rew + loss_indep_weighted

        if epoch == 1 and len(buf_total) == 0:
            print(
                f"[{METHOD}] INDEP_WEIGHT={INDEP_WEIGHT} | "
                f"dyn={loss_dyn.item():.4f}, "
                f"rew={loss_rew.item():.4f}, "
                f"indep(weighted)={loss_indep_weighted.item():.4f}"
            )

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optim.step()

        buf_total.append(loss.detach())
        buf_dyn.append(loss_dyn.detach())
        buf_rew.append(loss_rew.detach())
        buf_indep.append(loss_indep_weighted.detach())

    epoch_dyn   = torch.stack(buf_dyn).mean().item()
    epoch_rew   = torch.stack(buf_rew).mean().item()
    epoch_indep = torch.stack(buf_indep).mean().item()
    epoch_total = torch.stack(buf_total).mean().item()

    pretrain_history.append({
        "epoch":      epoch,
        "loss_total": epoch_total,
        "loss_dyn":   epoch_dyn,
        "loss_rew":   epoch_rew,
        "loss_indep": epoch_indep,  # 已乘 INDEP_WEIGHT
    })

    print(
        f"[pretrain] epoch {epoch} | "
        f"total={epoch_total:.4f}  "
        f"dyn={epoch_dyn:.4f}  "
        f"rew={epoch_rew:.4f}  "
        f"indep={epoch_indep:.4f}"
    )

CKPT_ENCODER = CKPT_DIR / f"encoder_epoch_{PRETRAIN_EPOCHS}.pth"
torch.save(encoder.state_dict(), CKPT_ENCODER)
print("Saved encoder:", CKPT_ENCODER)

In [ ]:
# Load and freeze encoder, then train IQL
encoder = load_and_freeze_encoder(
    encoder=encoder,
    ckpt_path=CKPT_ENCODER,
    device=DEVICE,
)

iql = IQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

iql_history = train_iql_from_loader(
    iql=iql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=encoder,
    repr_mode="disentangled",
    use_tqdm=False,
)

In [ ]:
# Evaluate and save metrics
print("Start evaluating ...")
metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        "indep_weight":       INDEP_WEIGHT,
        "latent_dim":         LATENT_DIM,
        "pretrain_epochs":    PRETRAIN_EPOCHS,
        "iql_epochs":         EPOCHS,
        "encoder_checkpoint": str(CKPT_ENCODER),
        "obs_stats_path":     str(OBS_DIR / "obs_stats.npz"),
        "pretrain_history":   pretrain_history,
        "iql_history":        iql_history,
    },
)

print("Saved metrics:", metrics_path)
print(metrics)